### staging dataset

In [1]:
from google.cloud import bigquery

project_id = "segfault-434120"
dataset = "skincare_products_stg"
region = "us-central1"

bq_client = bigquery.Client()

dataset_id = bigquery.Dataset(f"{project_id}.{dataset}")
dataset_id.location = region
resp = bq_client.create_dataset(dataset_id, exists_ok=True)
print("Created dataset {}.{}".format(bq_client.project, resp.dataset_id))

Created dataset segfault-434120.skincare_products_stg


In [ ]:
!pip install vertexai

### `cosmetic_ingredient` table

##### Replace empty or empty string equivalents with null

In [2]:
%%bigquery
select
  case cosing_ref_no when '' then null else cosing_ref_no end as cosing_ref_no,
  case ingredient_unique_name when '' then null else ingredient_unique_name end as ingredient_unique_name,
  case ingredient_common_name when '' then null else ingredient_common_name end as ingredient_common_name,
  case european_pharmacopoeia_name when '' then null else european_pharmacopoeia_name end as european_pharmacopoeia_name,
  case
    when cas_no in (' ', ' -') then null
    else cas_no
  end as cas_no,
  case
    when ec_no in (' ', ' ,', ' -', ' - ', ' - / -') then null
    else ec_no
  end as ec_no,
  case restriction when '' then null else restriction end as restriction,
  case function when '' then null else function end as function,
  case update_date when '' then null else update_date end as update_date,
from skincare_products_raw.cosmetic_ingredient;

Query is running:   0%|          |

Downloading:   0%|          |

,cosing_ref_no,ingredient_unique_name,ingredient_common_name,european_pharmacopoeia_name,cas_no,ec_no,restriction,function,update_date
0,None,None,None,None,None,None,None,None,None
1,92615,BIORESMETHRIN,None,None,28434-01-7,249-014-0 (I),None,None,21/07/2015
2,89117,CAPRYLYL LAURATE,None,None,5303-24-2,226-149-3,None,None,24/07/2012
3,88730,CARRAGEENAN,None,None,9000-07-1,232-524-2 (I),None,None,24/07/2012
4,89008,CITRUS AURANTIUM BERGAMIA PEEL OIL,None,None,89957-91-5,289-612-9,None,None,25/07/2012
...,...,...,...,...,...,...,...,...,...
28709,60168,WHEAT GERM OIL PEG-8 ESTERS,None,None,None,None,None,"SKIN CONDITIONING, SKIN CONDITIONING - EMOLLIE...",15/10/2010
28710,59648,SODIUM STEAROYL HYDROLYZED SWEET ALMOND PROTEIN,None,None,None,None,None,"ANTISTATIC, SKIN CONDITIONING, SKIN CONDITIONI...",15/10/2010
28711,82225,SODIUM HYDROXYPROPYLSULFONATE DECYLGLUCOSIDE C...,None,None,None,None,None,"CLEANSING, SURFACTANT - CLEANSING, SURFACTANT ...",15/10/2010
28712,57270,PEG/PPG-70/30 TOCOPHERYL ETHER,None,None,None,None,None,"ANTICAKING, ANTIOXIDANT, BINDING, EMULSION STA...",24/01/2011


#### Cast `date` from STRING to DATE

In [3]:
%%bigquery
select update_date as orig_date, safe_cast(update_date as DATE) as new_date
from skincare_products_raw.cosmetic_ingredient
limit 5

Query is running:   0%|          |

Downloading:   0%|          |

,orig_date,new_date
0,None,NaT
1,21/07/2015,NaT
2,24/07/2012,NaT
3,24/07/2012,NaT
4,25/07/2012,NaT


In [4]:
%%bigquery
select update_date as orig_date, safe_cast(update_date as DATE format 'DD/MM/YYYY') as new_date
from skincare_products_raw.cosmetic_ingredient
limit 5

Query is running:   0%|          |

Downloading:   0%|          |

,orig_date,new_date
0,None,NaT
1,21/07/2015,2015-07-21
2,24/07/2012,2012-07-24
3,24/07/2012,2012-07-24
4,25/07/2012,2012-07-25


#### Cast `cosing_ref_no` from STRING to INTEGER

In [5]:
%%bigquery
select cosing_ref_no as orig_cosing_ref_no, safe_cast(cosing_ref_no as INTEGER) as new_cosing_ref_no
from skincare_products_raw.cosmetic_ingredient
limit 5

Query is running:   0%|          |

Downloading:   0%|          |

,orig_cosing_ref_no,new_cosing_ref_no
0,None,<NA>
1,92615,92615
2,89117,89117
3,88730,88730
4,89008,89008


#### Rename ambiguous fields

In [6]:
%%bigquery
select
  restriction as regulatory_restriction,
  function as intended_use,
from skincare_products_raw.cosmetic_ingredient;

Query is running:   0%|          |

Downloading:   0%|          |

,regulatory_restriction,intended_use
0,None,None
1,,
2,,
3,,
4,,
...,...,...
28709,,"SKIN CONDITIONING, SKIN CONDITIONING - EMOLLIE..."
28710,,"ANTISTATIC, SKIN CONDITIONING, SKIN CONDITIONI..."
28711,,"CLEANSING, SURFACTANT - CLEANSING, SURFACTANT ..."
28712,,"ANTICAKING, ANTIOXIDANT, BINDING, EMULSION STA..."


#### Putting it all together

In [7]:
%%bigquery
select
  safe_cast(cosing_ref_no as INTEGER) as cosing_ref_no,
  case ingredient_unique_name when '' then null else ingredient_unique_name end as ingredient_unique_name,
  case ingredient_common_name when '' then null else ingredient_common_name end as ingredient_common_name,
  case european_pharmacopoeia_name when '' then null else european_pharmacopoeia_name end as european_pharmacopoeia_name,
  case
    when cas_no in (' ', ' -') then null
    else cas_no
  end as cas_no,
  case
    when ec_no in (' ', ' ,', ' -', ' - ', ' - / -') then null
    else ec_no
  end as ec_no,
  case restriction when '' then null else restriction end as regulatory_restriction,
  case function when '' then null else function end as intended_use,
  case update_date when '' then null else safe_cast(update_date as DATE format 'DD/MM/YYYY') end as update_date,
from skincare_products_raw.cosmetic_ingredient;

Query is running:   0%|          |

Downloading:   0%|          |

,cosing_ref_no,ingredient_unique_name,ingredient_common_name,european_pharmacopoeia_name,cas_no,ec_no,regulatory_restriction,intended_use,update_date
0,<NA>,None,None,None,None,None,None,None,NaT
1,92615,BIORESMETHRIN,None,None,28434-01-7,249-014-0 (I),None,None,2015-07-21
2,89117,CAPRYLYL LAURATE,None,None,5303-24-2,226-149-3,None,None,2012-07-24
3,88730,CARRAGEENAN,None,None,9000-07-1,232-524-2 (I),None,None,2012-07-24
4,89008,CITRUS AURANTIUM BERGAMIA PEEL OIL,None,None,89957-91-5,289-612-9,None,None,2012-07-25
...,...,...,...,...,...,...,...,...,...
28709,60168,WHEAT GERM OIL PEG-8 ESTERS,None,None,None,None,None,"SKIN CONDITIONING, SKIN CONDITIONING - EMOLLIE...",2010-10-15
28710,59648,SODIUM STEAROYL HYDROLYZED SWEET ALMOND PROTEIN,None,None,None,None,None,"ANTISTATIC, SKIN CONDITIONING, SKIN CONDITIONI...",2010-10-15
28711,82225,SODIUM HYDROXYPROPYLSULFONATE DECYLGLUCOSIDE C...,None,None,None,None,None,"CLEANSING, SURFACTANT - CLEANSING, SURFACTANT ...",2010-10-15
28712,57270,PEG/PPG-70/30 TOCOPHERYL ETHER,None,None,None,None,None,"ANTICAKING, ANTIOXIDANT, BINDING, EMULSION STA...",2011-01-24


#### Create staging table

In [9]:
%%bigquery
create or replace table skincare_products_stg.cosmetic_ingredient as
  select
  safe_cast(cosing_ref_no as INTEGER) as cosing_ref_no,
  case ingredient_unique_name when '' then null else ingredient_unique_name end as ingredient_unique_name,
  case ingredient_common_name when '' then null else ingredient_common_name end as ingredient_common_name,
  case european_pharmacopoeia_name when '' then null else european_pharmacopoeia_name end as european_pharmacopoeia_name,
  case
    when cas_no in (' ', ' -') then null
    else cas_no
  end as cas_no,
  case
    when ec_no in (' ', ' ,', ' -', ' - ', ' - / -') then null
    else ec_no
  end as ec_no,
  case restriction when '' then null else restriction end as regulatory_restriction,
  case function when '' then null else function end as intended_use,
  case update_date when '' then null else safe_cast(update_date as DATE format 'DD/MM/YYYY') end as update_date,
  _data_source,
   _load_time
  from skincare_products_raw.cosmetic_ingredient;

Query is running:   0%|          |

""


### `dermatologic_disease` table

#### Create staging table and format `treatments` into string list using LLM

In [108]:
import vertexai
from vertexai.generative_models import GenerativeModel, Part, HarmCategory, HarmBlockThreshold
from google.cloud import bigquery
import pandas as pd
import json
from pandas_gbq import read_gbq, to_gbq

# Initialize Vertex AI
project_id = "segfault-434120"
region = "us-central1"
model_name = "gemini-1.5-flash-001"
vertexai.init(project=project_id, location=region)
model = GenerativeModel(model_name)

# Initialize BigQuery client
bq_client = bigquery.Client()

# SQL query to pull treatments column from BigQuery
query = """select * from skincare_products_raw.dermatologic_disease"""

# Prompt for Vertex AI
prompt = """Go through this treatment string and look for ingredient-based treatments only.
Ingredient-based treatments are those which refer specifically to substances or compounds used in the treatment, such as "minocycline", "aloe", "vitamin E", or "salicylic acid".
Do not include general treatments like "surgery", "physical therapy", or "laser treatment".
Add only ingredient-based treatments into a list.
For example, if "minocycline", "aloe", "warm compresses", and "surgery" are included in the given set of treatments, create a list with "minocycline" and "aloe".
Do not include the others.

Return the original treatment string along with the list of ingredient-based treatments you extracted.
Format the results as JSON with the schema: original_treatments:string, ingredient_treatments:array<string>.
Do not include an explanation in your response.
"""

# Function to process treatment using Vertex AI
def process_treatment(treatment):
  try:
    resp = model.generate_content(contents=[treatment, prompt], safety_settings={
                HarmCategory.HARM_CATEGORY_DANGEROUS_CONTENT: HarmBlockThreshold.BLOCK_ONLY_HIGH,
                HarmCategory.HARM_CATEGORY_HATE_SPEECH: HarmBlockThreshold.BLOCK_ONLY_HIGH,
                HarmCategory.HARM_CATEGORY_HARASSMENT: HarmBlockThreshold.BLOCK_ONLY_HIGH,
                HarmCategory.HARM_CATEGORY_SEXUALLY_EXPLICIT: HarmBlockThreshold.BLOCK_ONLY_HIGH,
            })
    resp_text = resp.text.replace("```json", "").replace("```", "").replace("\n", "")
    return json.loads(resp_text)
  except:
    return {  "original_treatments": treatment,  "ingredient_treatments": []}

# Fetch data from BigQuery
# rows = bq_client.query(treatments_query).to_dataframe()
rows = read_gbq(query, project_id=project_id)

# Process the treatments column only if it is non-null
processed_treatments = []
for index, row in rows.iterrows():
    treatment_string = row["treatments"]

    if pd.notnull(treatment_string):
      processed = process_treatment(treatment_string)
      print(processed)
      # Update the treatments column with the processed ingredient list (as JSON string)
      rows.at[index, "treatments"] = json.dumps(processed["ingredient_treatments"])

# Load the transformed data back into BigQuery using pandas-gbq
table_id = f"{project_id}.skincare_products_stg.dermatologic_disease"
to_gbq(rows, table_id, project_id=project_id, if_exists='replace')

Downloading: 100%|██████████|
{'original_treatments': 'Cocoa butter, shea butter, vitamin E, gotu kola extract, hyaluronic acid, elastin, egg oil, wheat germ oil, retinoids, vitamin C, alpha hydroxy acids (glycolic, lactic, citric, tartaric, and malic acids),  trichloroacetic acid (TCA), and microdermabrasion.', 'ingredient_treatments': ['Cocoa butter', 'shea butter', 'vitamin E', 'gotu kola extract', 'hyaluronic acid', 'elastin', 'egg oil', 'wheat germ oil', 'retinoids', 'vitamin C', 'glycolic acid', 'lactic acid', 'citric acid', 'tartaric acid', 'malic acid', 'trichloroacetic acid (TCA)']}
{'original_treatments': 'The main treatment focuses on preventing the wet-to-dry cycles. This involves wearing breathable footwear, thicker absorbent socks, and avoiding rapid drying of the feet. An occlusive, soothing ointment is applied to the affected areas to prevent drying out.', 'ingredient_treatments': []}
{'original_treatments': 'Surgical excision is the most common treatment for moles. Sma

100%|██████████| 1/1 [00:00<00:00, 6141.00it/s]


In [115]:
%%bigquery
create or replace table skincare_products_stg.dermatologic_disease as
  select name, physical_description, causes,
  JSON_EXTRACT_ARRAY(treatments) as treatments,
  _data_source,
  _load_time
  from skincare_products_stg.dermatologic_disease;

Query is running:   0%|          |

""


#### Standardize treatment ingredients using LLM

##### View list of treatment ingredients

In [121]:
%%bigquery
WITH flattened_treatments AS (
  -- Flatten the array into individual rows
  SELECT
    LOWER(TRIM(treatment)) AS treatment -- Convert to lowercase and trim spaces to handle case/whitespace variations
  FROM
    skincare_products_stg.dermatologic_disease,
    UNNEST(treatments) AS treatment -- Unnest the arrays into individual strings
  WHERE
    treatments IS NOT NULL
)
-- Select distinct treatments and sort them alphabetically
SELECT DISTINCT treatment as ingredient
FROM flattened_treatments
ORDER BY ingredient;

Query is running:   0%|          |

Downloading:   0%|          |

,ingredient
0,"""5-fluorouracil"""
1,"""5-fu"""
2,"""abrocitinib"""
3,"""accolate"""
4,"""accutane"""
...,...
324,"""wheat germ oil"""
325,"""zinc oxide"""
326,"""zinc pyrithione"""
327,"""zinc sulfate"""


##### Create standardized ingredient names

In [124]:
import json
import pandas_gbq
import numpy as np
import vertexai
from vertexai.generative_models import GenerativeModel, Part
from google.cloud import bigquery
import pandas as pd
import ast


project_id = "segfault-434120"
region = "us-central1"
dataset_id = "skincare_products_stg"
model_name = "gemini-1.5-flash-001"
prompt = """Go through this list of ingredients and look for ingredients that have similar meanings, but were given different names.
For example, '5-FU' and '5-fluorouracil' have similar meanings and 'alpha-hydroxy acid' and 'alpha hydroxyacids' have similar meanings.
Suggest a standard ingredient name, mapping the current one to the new one.
Return the list of original ingredient names along with their new ingredient names.
Format the results as a json object with the schema: current_ingredient:string, new_ingredient:string.
Do not include any unchanged ingredient names with your answer.
Do not include an explanation with your answer.
"""

ingredients_sql = """
WITH flattened_treatments AS (
  -- Flatten the array into individual rows
  SELECT
    LOWER(TRIM(treatment)) AS treatment -- Convert to lowercase and trim spaces to handle case/whitespace variations
  FROM
    skincare_products_stg.dermatologic_disease,
    UNNEST(treatments) AS treatment -- Unnest the arrays into individual strings
  WHERE
    treatments IS NOT NULL
)
-- Select distinct treatments and sort them alphabetically
SELECT DISTINCT treatment as ingredient
FROM flattened_treatments
ORDER BY ingredient;
"""

dermatologic_disease_sql = "select * from skincare_products_stg.dermatologic_disease"

bq_client = bigquery.Client()
ingredient_rows = bq_client.query_and_wait(ingredients_sql)

ingredient_list = []
for row in ingredient_rows:
    ingredient_list.append(row["ingredient"])
category_str = '\n'.join(ingredient_list)
#print("category_str:", category_str)

vertexai.init(project=project_id, location=region)
model = GenerativeModel(model_name)
resp = model.generate_content(contents=[category_str, prompt], safety_settings={
                HarmCategory.HARM_CATEGORY_DANGEROUS_CONTENT: HarmBlockThreshold.BLOCK_ONLY_HIGH,
                HarmCategory.HARM_CATEGORY_HATE_SPEECH: HarmBlockThreshold.BLOCK_ONLY_HIGH,
                HarmCategory.HARM_CATEGORY_HARASSMENT: HarmBlockThreshold.BLOCK_ONLY_HIGH,
                HarmCategory.HARM_CATEGORY_SEXUALLY_EXPLICIT: HarmBlockThreshold.BLOCK_ONLY_HIGH,
            })
resp_text = resp.text.replace("```json", "").replace("```", "").replace("\n", "")
print(resp_text)
ingredients = json.loads(resp_text)
print(type(ingredients))

replacements = {} # will store the new ingredients

# ingredients can be either a dictionary or list type (depending on what LLM decides to do!)
if type(ingredients) == dict:
    for old, new in ingredients.items():
        if old == new:
            continue
        else:
            replacements[old] = new

if type(ingredients) == list:
    for cat_entry in ingredients:
        if cat_entry['current_ingredient'] == cat_entry['new_ingredient']:
            continue
        else:
            replacements[cat_entry['current_ingredient']] = cat_entry['new_ingredient']


# lowercase all ingredient names
replacements = {key.lower(): value.lower() for key, value in replacements.items()}
print("replacements:", replacements)

{"5-fu": "5-fluorouracil","alpha hydroxyacids": "alpha-hydroxy acid","alpha hydroxyl acids": "alpha-hydroxy acid","anti-bacterials": "antibiotics","anti-dandruff shampoo": "selenium sulfide","anti-fungal creams": "clotrimazole","anti-fungal powder": "clotrimazole","anti-inflammatory pills": "ibuprofen","botulinum toxin": "botulinum toxin a","calcineurin inhibitors": "tacrolimus","calcipotriol": "calcipotriene","calcium channel blockers": "nifedipine","capsaicin": "capasaicin","cephalosporin": "cephalosporins","chlorine bleach": "bleach","cleocin-t": "clindamycin","clindamycin hcl": "clindamycin","coal tar": "tar","corticosteroid medications": "corticosteroids","corticosteroid": "corticosteroids","cortisone": "corticosteroids","cromolyn": "cromolyn sodium","cyclosporin": "cyclosporine","cytoxan": "cyclophosphamide","diclofenac sodium": "diclofenac","diosmiplex": "diosmin","doxycycline": "tetracyclines","elavil": "amitriptyline","famvir": "famciclovir","fluids": "electrolytes","fluoroura

##### Replace treatment ingredient names with their standardized name

In [125]:
# Function to replace ingredients in each treatment list
def replace_ingredients(treatments, replacements_dict):
    print("original treatments: ", treatments)

    # Check if treatments is NULL or not a list (e.g., None)
    if treatments is None or not isinstance(treatments, list):
        return treatments

    # Check if treatments array is empty
    if len(treatments) == 0:
        return treatments

    standardized_treatments = []

    # Iterate through each ingredient in the treatments array
    treatments = [x.lower() for x in treatments]
    for ingredient in treatments:
        ingredient = ingredient.replace("\"", "")

        # Replace with standardized name if it exists in the replacements dictionary
        standardized_ingredient = replacements_dict.get(ingredient, ingredient)
        standardized_treatments.append(standardized_ingredient)

    print("standardized_treatments: ", standardized_treatments)
    return standardized_treatments

# Function to safely apply literal_eval only when needed
def ensure_list(val):
    # If the value is a NumPy array, convert it to a list
    if isinstance(val, np.ndarray):
        # print("original nparray val: ", val)
        val_as_list = val.tolist()
        # print("list val: ", val_as_list)

        # If tolist() returns a string or other non-list type, wrap it in a list
        if isinstance(val_as_list, str):
            # print("wrapped val: ", [val_as_list])
            return [val_as_list]

        return val_as_list

# Fetch the data from BigQuery into a DataFrame
disease_rows = pandas_gbq.read_gbq(dermatologic_disease_sql, project_id=project_id)

# Convert lists to JSON strings for BigQuery compatibility
disease_rows["treatments"] = disease_rows["treatments"].apply(ensure_list)
disease_rows["treatments"] = disease_rows["treatments"].apply(lambda x: replace_ingredients(x, replacements))
disease_rows["treatments"] = disease_rows["treatments"].apply(lambda x: json.dumps(x) if isinstance(x, list) else x)

# # Apply the replacement to each row's treatments list
# print("new df:", disease_rows)

# # Upload the updated DataFrame back to BigQuery
table_id = f"{project_id}.skincare_products_stg.dermatologic_disease"
pandas_gbq.to_gbq(disease_rows, table_id, project_id=project_id, if_exists='replace')

Downloading: 100%|██████████|
original treatments:  []
original treatments:  []
original treatments:  []
original treatments:  []
original treatments:  []
original treatments:  []
original treatments:  []
original treatments:  []
original treatments:  []
original treatments:  []
original treatments:  []
original treatments:  []
original treatments:  []
original treatments:  []
original treatments:  []
original treatments:  []
original treatments:  []
original treatments:  []
original treatments:  []
original treatments:  []
original treatments:  []
original treatments:  []
original treatments:  []
original treatments:  []
original treatments:  []
original treatments:  []
original treatments:  []
original treatments:  []
original treatments:  []
original treatments:  []
original treatments:  []
original treatments:  []
original treatments:  []
original treatments:  []
original treatments:  []
original treatments:  []
original treatments:  []
original treatments:  []
original treatments:

100%|██████████| 1/1 [00:00<00:00, 8176.03it/s]


##### Cast treatments from STRING to ARRAY<STRING>

In [129]:
%%bigquery
SELECT
  -- Use JSON_EXTRACT_ARRAY to convert the cleaned string into an array
  JSON_EXTRACT_ARRAY(treatments) AS treatments
FROM
  `skincare_products_stg.dermatologic_disease`

Query is running:   0%|          |

Downloading:   0%|          |

,treatments
0,[]
1,[]
2,[]
3,[]
4,[]
...,...
365,"[""corticosteroids"", ""pimecrolimus"", ""tacrolimu..."
366,"[""tetracyclines"", ""tetracyclines"", ""corticoste..."
367,"[""methotrexate"", ""acitretin"", ""cyclosporine"", ..."
368,"[""tetracyclines"", ""salicylic acid"", ""benzoyl p..."


#### Replace `N/A` and `Not applicable` with NULL

In [130]:
%%bigquery
select
case
  when causes in ('N/A', 'Not applicable') then null
  else causes
  end as causes,
from skincare_products_stg.dermatologic_disease;

Query is running:   0%|          |

Downloading:   0%|          |

,causes
0,"The cause of Spitz nevus is currently unknown,..."
1,A normal physiologic response of the newborn t...
2,Dermoscopy is used to evaluate both benign and...
3,"The exact cause of psoriasis is unknown, but i..."
4,Unwanted hair growth can be caused by several ...
...,...
365,"The exact cause of DLE is unknown, but it is t..."
366,An autoimmune disorder where the body's immune...
367,Psoriasis is triggered by factors such as bact...
368,The disease is spread by ticks that acquire th...


#### Putting it all together

In [131]:
%%bigquery
select name, physical_description,
case
  when causes in ('N/A', 'Not applicable') then null
  else causes
  end as causes,
JSON_EXTRACT_ARRAY(treatments) as treatments,
_data_source,
_load_time
from skincare_products_stg.dermatologic_disease;

Query is running:   0%|          |

Downloading:   0%|          |

,name,physical_description,causes,treatments,_data_source,_load_time
0,Spitz Nevus,"Spitz nevus is typically a solitary, round, fi...","The cause of Spitz nevus is currently unknown,...",[],dermatologic_disease,2024-09-25 02:15:27.599878+00:00
1,Cutis Marmorata,"A red and/or blue, lacy pattern appears on the...",A normal physiologic response of the newborn t...,[],dermatologic_disease,2024-09-25 02:15:27.599878+00:00
2,Dermoscopy,Dermoscopy is a non-invasive technique used to...,Dermoscopy is used to evaluate both benign and...,[],dermatologic_disease,2024-09-25 02:15:27.599878+00:00
3,Psoriasis,"Psoriasis is a skin condition that causes red,...","The exact cause of psoriasis is unknown, but i...",[],dermatologic_disease,2024-09-25 02:15:27.599878+00:00
4,Unwanted Hair Growth,Unwanted hair growth occurs when hair grows in...,Unwanted hair growth can be caused by several ...,[],dermatologic_disease,2024-09-25 02:15:27.599878+00:00
...,...,...,...,...,...,...
365,Discoid Lupus Erythematosus,Discoid lupus erythematosus (DLE) is a chronic...,"The exact cause of DLE is unknown, but it is t...","[""corticosteroids"", ""pimecrolimus"", ""tacrolimu...",dermatologic_disease,2024-09-25 02:15:27.599878+00:00
366,Bullous Pemphigoid,A chronic blistering skin condition that can r...,An autoimmune disorder where the body's immune...,"[""tetracyclines"", ""tetracyclines"", ""corticoste...",dermatologic_disease,2024-09-25 02:15:27.599878+00:00
367,Psoriasis,Psoriasis is a common skin condition character...,Psoriasis is triggered by factors such as bact...,"[""methotrexate"", ""acitretin"", ""cyclosporine"", ...",dermatologic_disease,2024-09-25 02:15:27.599878+00:00
368,Lyme Disease,"Flu-like symptoms, fever, muscle aches, joint ...",The disease is spread by ticks that acquire th...,"[""tetracyclines"", ""salicylic acid"", ""benzoyl p...",dermatologic_disease,2024-09-25 02:15:27.599878+00:00


#### Apply changes to staging table

In [132]:
%%bigquery
create or replace table skincare_products_stg.dermatologic_disease as
  select name, physical_description,
  case
    when causes in ('N/A', 'Not applicable') then null
    else causes
    end as causes,
  JSON_EXTRACT_ARRAY(treatments) as treatments,
  _data_source,
  _load_time
  from skincare_products_stg.dermatologic_disease;

Query is running:   0%|          |

""


### `sephora_product` table

#### Convert `limited_edition`, `new`, `online_only`, `out_of_stock`, and `sephora_exclusive` from Integer to Bool

In [28]:
%%bigquery
select
  safe_cast(limited_edition as BOOLEAN) as limited_edition,
  safe_cast(`new` as BOOLEAN) as `new`,
  safe_cast(online_only as BOOLEAN) as online_only,
  safe_cast(out_of_stock as BOOLEAN) as out_of_stock,
  safe_cast(sephora_exclusive as BOOLEAN) as sephora_exclusive
from skincare_products_raw.sephora_product

Query is running:   0%|          |

Downloading:   0%|          |

,limited_edition,new,online_only,out_of_stock,sephora_exclusive
0,False,False,True,True,True
1,False,False,True,False,False
2,False,False,True,True,True
3,False,False,False,False,True
4,False,True,True,False,False
...,...,...,...,...,...
8489,False,False,True,False,False
8490,False,False,False,False,False
8491,False,False,True,False,False
8492,False,True,True,True,True


#### Rename ambigious fields

In [29]:
%%bigquery
select
  `new` as new_product,
  child_count as variation_count,
  child_max_price as variation_max_price,
  child_min_price as variation_min_price,
  highlights as feature_highlights
from skincare_products_raw.sephora_product

Query is running:   0%|          |

Downloading:   0%|          |

,new_product,variation_count,variation_max_price,variation_min_price,feature_highlights
0,0,0,NaN,NaN,"['Good for: Dryness', 'Clean at Sephora', 'Hyd..."
1,0,0,NaN,NaN,"['Good for: Dryness', 'Clean at Sephora', 'Hyd..."
2,0,0,NaN,NaN,"['Hydrating', 'Good for: Frizz', 'Good for: Dr..."
3,0,0,NaN,NaN,"['Vegan', 'Good for: Loss of firmness', 'Good ..."
4,1,0,NaN,NaN,"['Vegan', 'Good for: Loss of firmness', 'Plump..."
...,...,...,...,...,...
8489,0,0,NaN,NaN,"['allure 2022 Best of Beauty Award Winner', 'C..."
8490,0,0,NaN,NaN,None
8491,0,0,NaN,NaN,"['Stick Formula', 'Floral Scent', 'Alcohol Fre..."
8492,1,0,NaN,NaN,"['Vegan', 'Good for: Dullness/Uneven Texture',..."


#### Split `size` into imperial_size, metric_size, and other_size

In [58]:
%%bigquery
select distinct
  case
    when size not like '%oz%'
      and size not like '%g%'
      and size not like '%mL%'
      and size not like '%ml%'
      and size not like '%in%'
      and size not like '%inches%'
      and size not like '%Inches%'
      and size not like '%/%'
    then size
    else null
  end as other_size,
  case
    when contains_substr(size, '/') then split(size, '/')[SAFE_OFFSET(0)]
    when size like '%oz%'
      or size like '%inches%'
      or size like '%Inches%'
    then size
    else null
  end as imperial_size,
  case
    when contains_substr(size, '/') then split(size, '/')[SAFE_OFFSET(1)]
    when size like '%g%'
      or size like '%mL%'
      or size like '%ml%'
    then size
    else null
  end as metric_size,
from skincare_products_raw.sephora_product

Query is running:   0%|          |

Downloading:   0%|          |

,other_size,imperial_size,metric_size
0,None,None,None
1,None,8 x .02 oz,.80 g
2,"1"" wand",None,None
3,None,2 x .032 oz,0.9 g
4,None,2 oz,60 mL
...,...,...,...
2046,None,0.33 oz,9.8 mL eau de parfum spray
2047,None,.33oz,10mL
2048,None,0.3 oz,7.5 mL
2049,None,0.34 oz,10 mL Perfume Spray


#### Extracting color from variation_type and variation_value

In [75]:
%%bigquery
select variation_type,
  case when variation_type like '%Color%' then variation_value else null
  end as color
from skincare_products_raw.sephora_product
group by variation_type, variation_value
order by variation_type

Query is running:   0%|          |

Downloading:   0%|          |

,variation_type,color
0,None,None
1,Color,Lilac
2,Color,Taupe
3,Color,Medium Brown
4,Color,Tease Satin Lipstick + Baby Roses Mini Lip Liner
...,...,...
2814,Type,None
2815,Type,None
2816,Type,None
2817,Type,None


#### Putting it all together

In [133]:
%%bigquery
select
  product_id,
  product_name,
  brand_id,
  brand_name,
  loves_count,
  rating,
  reviews,
  case
    when size not like '% oz%'
      and size not like '% g%'
      and size not like '% mL%'
      and size not like '% ml%'
      and size not like '% in%'
      and size not like '% inches%'
      and size not like '% Inches%'
      and size not like '%/%'
    then size
    else null
  end as other_size,
  case
    when contains_substr(size, '/') then split(size, '/')[SAFE_OFFSET(0)]
    when size like '% oz%'
      or size like '% inches%'
      or size like '% Inches%'
    then size
    else null
  end as imperial_size,
  case
    when contains_substr(size, '/') then split(size, '/')[SAFE_OFFSET(1)]
    when size like '% g%'
      or size like '% mL%'
      or size like '% ml%'
    then size
    else null
  end as metric_size,
  case
    when variation_type like '%Color%' then variation_value
    else null
  end as color,
  child_count as variation_count,
  child_max_price as variation_max_price,
  child_min_price as variation_min_price,
  JSON_EXTRACT_ARRAY(ingredients) as ingredients,
  price_usd,
  value_price_usd,
  sale_price_usd,
  safe_cast(limited_edition as BOOLEAN) as limited_edition,
  safe_cast(`new` as BOOLEAN) as new_product,
  safe_cast(online_only as BOOLEAN) as online_only,
  safe_cast(out_of_stock as BOOLEAN) as out_of_stock,
  safe_cast(sephora_exclusive as BOOLEAN) as sephora_exclusive,
  highlights as feature_highlights,
  primary_category,
  secondary_category,
  tertiary_category,
  _data_source,
  _load_time
from skincare_products_raw.sephora_product;

Query is running:   0%|          |

Downloading:   0%|          |

,product_id,product_name,brand_id,brand_name,loves_count,rating,reviews,other_size,imperial_size,metric_size,...,new_product,online_only,out_of_stock,sephora_exclusive,feature_highlights,primary_category,secondary_category,tertiary_category,_data_source,_load_time
0,P476418,African Beauty Butter Mini Gift Set,6471,54 Thrones,7526,3.5610,41,None,None,None,...,False,True,True,True,"['Good for: Dryness', 'Clean at Sephora', 'Hyd...",Bath & Body,Value & Gift Sets,None,sephora_product,2024-09-25 02:15:33.136361+00:00
1,P476417,African Beauty Butter Collection Deluxe Tin,6471,54 Thrones,3741,4.2273,22,None,None,None,...,False,True,False,False,"['Good for: Dryness', 'Clean at Sephora', 'Hyd...",Bath & Body,Value & Gift Sets,None,sephora_product,2024-09-25 02:15:33.136361+00:00
2,P473151,Mini Baomint Deluxe Travel Kit,6321,adwoa beauty,4559,4.7283,173,None,None,None,...,False,True,True,True,"['Hydrating', 'Good for: Frizz', 'Good for: Dr...",Hair,Value & Gift Sets,None,sephora_product,2024-09-25 02:15:33.136361+00:00
3,P500716,10 Day Results Kit,6018,Algenist,3206,4.8023,258,None,None,None,...,False,False,False,True,"['Vegan', 'Good for: Loss of firmness', 'Good ...",Skincare,Value & Gift Sets,None,sephora_product,2024-09-25 02:15:33.136361+00:00
4,P504443,Plump Pout Duo,6018,Algenist,522,3.0000,1,None,None,None,...,True,True,False,False,"['Vegan', 'Good for: Loss of firmness', 'Plump...",Skincare,Value & Gift Sets,None,sephora_product,2024-09-25 02:15:33.136361+00:00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8489,P481959,Wake the F*ck Up Deodorant,8007,By Rosie Jane,7318,3.9773,44,None,2oz,59mL,...,False,True,False,False,"['allure 2022 Best of Beauty Award Winner', 'C...",Bath & Body,Body Care,Deodorant & Antiperspirant,sephora_product,2024-09-25 02:15:33.136361+00:00
8490,P473115,BLEU DE CHANEL All-over Spray,1065,CHANEL,998,NaN,<NA>,None,5 oz,148 mL,...,False,False,False,False,None,Bath & Body,Body Care,Deodorant & Antiperspirant,sephora_product,2024-09-25 02:15:33.136361+00:00
8491,P503730,Cashmere Mist Aluminum Free/Alcohol Free Deodo...,3491,Donna Karan,3171,3.3333,15,None,1.7 oz,50 mL,...,False,True,False,False,"['Stick Formula', 'Floral Scent', 'Alcohol Fre...",Bath & Body,Body Care,Deodorant & Antiperspirant,sephora_product,2024-09-25 02:15:33.136361+00:00
8492,P504143,Rio Deo Aluminum-Free Deodorant Cheirosa '40,6145,Sol de Janeiro,20982,3.4444,63,None,2 oz,57 g,...,True,True,True,True,"['Vegan', 'Good for: Dullness/Uneven Texture',...",Bath & Body,Body Care,Deodorant & Antiperspirant,sephora_product,2024-09-25 02:15:33.136361+00:00


#### Create staging table

In [134]:
%%bigquery
create or replace table skincare_products_stg.sephora_product as
  select
    product_id,
    product_name,
    brand_id,
    brand_name,
    loves_count,
    rating,
    reviews,
    case
      when size not like '% oz%'
        and size not like '% g%'
        and size not like '% mL%'
        and size not like '% ml%'
        and size not like '% in%'
        and size not like '% inches%'
        and size not like '% Inches%'
        and size not like '%/%'
      then size
      else null
    end as other_size,
    case
      when contains_substr(size, '/') then split(size, '/')[SAFE_OFFSET(0)]
      when size like '% oz%'
        or size like '% inches%'
        or size like '% Inches%'
      then size
      else null
    end as imperial_size,
    case
      when contains_substr(size, '/') then split(size, '/')[SAFE_OFFSET(1)]
      when size like '% g%'
        or size like '% mL%'
        or size like '% ml%'
      then size
      else null
    end as metric_size,
    case
      when variation_type like '%Color%' then variation_value
      else null
    end as color,
    child_count as variation_count,
    child_max_price as variation_max_price,
    child_min_price as variation_min_price,
    JSON_EXTRACT_ARRAY(ingredients) as ingredients,
    price_usd,
    value_price_usd,
    sale_price_usd,
    safe_cast(limited_edition as BOOLEAN) as limited_edition,
    safe_cast(`new` as BOOLEAN) as new_product,
    safe_cast(online_only as BOOLEAN) as online_only,
    safe_cast(out_of_stock as BOOLEAN) as out_of_stock,
    safe_cast(sephora_exclusive as BOOLEAN) as sephora_exclusive,
    highlights as feature_highlights,
    primary_category,
    secondary_category,
    tertiary_category,
    _data_source,
    _load_time
  from skincare_products_raw.sephora_product

Query is running:   0%|          |

""


#### Standardize Ingredient using LLM

##### Fetch ingredients from `sephora_product` table

In [37]:
%%bigquery
WITH flattened_ingredients AS (
  -- Flatten the array into individual rows and split the ingredient strings
  SELECT
    LOWER(TRIM(ingredient_part)) AS ingredient -- Convert to lowercase and trim spaces
  FROM
    skincare_products_stg.sephora_product,
    UNNEST(ingredients) AS ingredient, -- Unnest the array into individual strings
    UNNEST(SPLIT(ingredient, ',')) AS ingredient_part -- Split each string by comma and unnest into individual parts
  WHERE
    ingredients IS NOT NULL
)
-- Select distinct ingredients and sort them alphabetically
SELECT DISTINCT ingredient
FROM flattened_ingredients
ORDER BY ingredient;

Query is running:   0%|          |

Downloading:   0%|          |

,ingredient
0,
1,""""
2,""""""
3,"""#12103/a alcohol"
4,"""#14881/a talc"
...,...
18814,zymomonas ferment extract
18815,¬glycyrrhiza glabra (licorice) root extract
18816,α-phellandrene
18817,ψ hydrolyzed quinoa


##### (LONG RUNTIME) Generate standardized replacements using LLM

In [39]:
import json
import pandas_gbq
import numpy as np
import vertexai
from vertexai.generative_models import GenerativeModel, Part, HarmCategory, HarmBlockThreshold
from google.cloud import bigquery
import pandas as pd
import ast


project_id = "segfault-434120"
region = "us-central1"
dataset_id = "skincare_products_stg"
model_name = "gemini-1.5-flash-001"
prompt = """Go through this list of ingredients and look for ingredients that have similar meanings, but were given different names.
For example, '5-FU' and '5-fluorouracil' have similar meanings and 'alpha-hydroxy acid' and 'alpha hydroxyacids' have similar meanings.
Suggest a standard ingredient name, mapping the current one to the new one.
The standard name should not have any arbitrary special characters.
For example, 'ψ hydrolyzed quinoa' should instead be 'hydrolyzed quinoa'.
Some names are product names rather than ingredient names, so map them to an empty string.
For example, 'your skin but better setting spray', 'you’re my sol mate', and 'witch hazel & yucca deep hydrating shampoo' should all be mapped to ''.
Return the list of original ingredient names along with their new ingredient names.
Format the results as a json object with the schema: current_ingredient:string, new_ingredient:string.
Do not include any unchanged ingredient names with your answer.
Do not include an explanation with your answer.
"""

reformat_prompt = """You are given the following JSON data which contains formatting issues and errors.
Your task is to correct the format and return a valid JSON object.
Make sure all necessary commas, quotes, and braces are correctly placed.
For example, all key-value pairs must follow the structure "key string":"value string" within the json brackets.
Do not confuse a : inside a string with the : separating the key-value pair of the dictionary.
For example, {"your skin but better setting spray+:""your skin but better setting spray"} is missing a colon between the key and value pair.
Format the results as a json object with the schema: current_ingredient:string, new_ingredient:string.
Do not change the actual data values or structure, only correct the formatting errors.
Do not include an explanation with your answer.
"""

product_ingredients_sql = """
WITH flattened_ingredients AS (
  -- Flatten the array into individual rows and split the ingredient strings
  SELECT
    LOWER(TRIM(ingredient_part)) AS ingredient -- Convert to lowercase and trim spaces
  FROM
    skincare_products_stg.sephora_product,
    UNNEST(ingredients) AS ingredient, -- Unnest the array into individual strings
    UNNEST(SPLIT(ingredient, ',')) AS ingredient_part -- Split each string by comma and unnest into individual parts
  WHERE
    ingredients IS NOT NULL
)
-- Select distinct ingredients and sort them alphabetically
SELECT DISTINCT ingredient
FROM flattened_ingredients
ORDER BY ingredient;
"""

bq_client = bigquery.Client()
ingredient_rows = bq_client.query_and_wait(product_ingredients_sql)

ingredient_list = []
for row in ingredient_rows:
    ingredient_list.append(row["ingredient"])


replacements = {} # will store the new ingredients

# Find replacements in chunks
CHUNKS = 70
ingredient_list_chunks = np.array_split(ingredient_list, CHUNKS)
for i, chunk in enumerate(ingredient_list_chunks):
  ingredient_str = '\n'.join(chunk)
  #print("ingredient_str:", ingredient_str)

  vertexai.init(project=project_id, location=region)
  model = GenerativeModel(model_name)
  try:
    resp = model.generate_content(contents=[ingredient_str, prompt], safety_settings={
                  HarmCategory.HARM_CATEGORY_DANGEROUS_CONTENT: HarmBlockThreshold.BLOCK_ONLY_HIGH,
                  HarmCategory.HARM_CATEGORY_HATE_SPEECH: HarmBlockThreshold.BLOCK_ONLY_HIGH,
                  HarmCategory.HARM_CATEGORY_HARASSMENT: HarmBlockThreshold.BLOCK_ONLY_HIGH,
                  HarmCategory.HARM_CATEGORY_SEXUALLY_EXPLICIT: HarmBlockThreshold.BLOCK_ONLY_HIGH,
              })
    resp_text = resp.text.replace("```json", "").replace("```", "").replace("\n", "")
    print(resp_text)
  except ValueError as ve:
    print(f"Content moderation error on chunk {i}: {ve}")
    continue  # Skip this chunk if moderation blocks the response

  max_retries = 3
  retry_count = 0
  status = ""
  success = False

  while retry_count < max_retries and not success:
      try:
          ingredients = json.loads(resp_text)
          success = True  # Exit the loop if parsing is successful
      except:
          print("INVALID JSON, reformatting...")

          # Attempt to fix JSON
          status = f"STATUS: This is reformatting attempt #{retry_count}. Make sure you aren't overlooking JSON errors. There may be more than one."
          try:
            resp = model.generate_content(contents=[status, resp_text, reformat_prompt], safety_settings={
                HarmCategory.HARM_CATEGORY_DANGEROUS_CONTENT: HarmBlockThreshold.BLOCK_ONLY_HIGH,
                HarmCategory.HARM_CATEGORY_HATE_SPEECH: HarmBlockThreshold.BLOCK_ONLY_HIGH,
                HarmCategory.HARM_CATEGORY_HARASSMENT: HarmBlockThreshold.BLOCK_ONLY_HIGH,
                HarmCategory.HARM_CATEGORY_SEXUALLY_EXPLICIT: HarmBlockThreshold.BLOCK_ONLY_HIGH,
            })
            resp_text = resp.text.replace("```json", "").replace("```", "").replace("\n", "")
            print(resp_text)

            # Attempt to load the reformatted JSON again
            try:
                ingredients = json.loads(resp_text)
                success = True  # Exit the loop if parsing is successful
            except:
                print(f"Retry {retry_count + 1} failed.")
                retry_count += 1
          except:
            print("Failed to generate JSON. Content Moderation Error")
            retry_count += 1



  if not success:
      print(f"Failed to parse JSON after {retry_count + 1} retries. Skipping chunk {i}")
  print(type(ingredients))


  # ingredients can be either a dictionary or list type (depending on what LLM decides to do!)
  if type(ingredients) == dict:
      for old, new in ingredients.items():
          if old == new:
              continue
          else:
              replacements[old] = new

  if type(ingredients) == list:
      for cat_entry in ingredients:
          if cat_entry['current_ingredient'] == cat_entry['new_ingredient']:
              continue
          else:
              replacements[cat_entry['current_ingredient']] = cat_entry['new_ingredient']


# lowercase all ingredient names
replacements = {key.lower(): value.lower() for key, value in replacements.items()}
print("replacements:", replacements)



{"(+/-) may contain/peut contenir: red 40 lake (ci 16035)": "red 40 lake","(+/-) may contain/peut contenir: iron oxides (ci 77491": "iron oxides","(+/-) may contain/peut contenir: iron oxides (ci 77492": "iron oxides","(+/-) may contain/peut contenir: iron oxides (ci 77492": "iron oxides","(ci 15850)": "red 6","(ci 75470)": "red 7","(ci 77163)": "bismuth oxychloride","(ci 77491)": "iron oxides","(ci 77499 / iron oxides)": "iron oxides","(ci 77499)": "iron oxides","(ci 77499)  ": "iron oxides","(ci 77510)": "ferric ferrocyanide","(ci 77742)].": "ultramarines","(ci 77891 (titanium dioxide)": "titanium dioxide","(ci 77891)": "titanium dioxide","(fragrance)": "parfum","(natural)": "","(only 0.005% glycerin).": "glycerin","(pisum sativum (pea) extract)": "pea extract","(re)setting mist:": "","(±) may contain/peut contenir: copper powder (ci 77400)": "copper powder","(±) may contain/peut contenir: iron oxides (ci 77492": "iron oxides","(±) may contain/peut contenir: iron oxides (ci 77492": "

##### Apply standardized replacements to ingredients

In [135]:
# Function to replace ingredients in each treatment list
def replace_ingredients(ingredient_str_list, replacements_dict):
    # print(ingredient_str_list)

    # Check if treatments is NULL or not a list (e.g., None)
    if ingredient_str_list is None or not isinstance(ingredient_str_list, list):
        return []

    # Check if treatments array is empty
    if len(ingredient_str_list) == 0:
        return []

    standardized_ingredients = []

    # Combine ingredients string into single list of ingredients
    ingredients = []
    for ingredient_str in ingredient_str_list:
      ingredients += ingredient_str.split(", ")

    # Iterate through each ingredient in the ingredients array
    ingredients = [x.lower().replace("\"", "").strip() for x in ingredients]
    for ingredient in ingredients:
        # Replace with standardized name if it exists in the replacements dictionary
        # if (ingredient in replacements_dict):
        #   print("REPLACED: ", ingredient, "WITH", replacements_dict[ingredient])
        standardized_ingredient = replacements_dict.get(ingredient, ingredient)
        standardized_ingredient = standardized_ingredient.replace("\"", "").strip()

        if (standardized_ingredient != ""):
          standardized_ingredients.append(standardized_ingredient)

    # print(ingredients)
    # print(standardized_ingredients)
    # print()

    return standardized_ingredients

# Function to safely apply literal_eval only when needed
def ensure_list(val):
    # If the value is a NumPy array, convert it to a list
    if isinstance(val, np.ndarray):
        # print("original nparray val: ", val)
        val_as_list = val.tolist()
        # print("list val: ", val_as_list)

        # If tolist() returns a string or other non-list type, wrap it in a list
        if isinstance(val_as_list, str):
            # print("wrapped val: ", [val_as_list])
            return [val_as_list]

        return val_as_list

# Fetch the data from BigQuery into a DataFrame
sephora_product_sql = "select * from skincare_products_stg.sephora_product"
product_rows = pandas_gbq.read_gbq(sephora_product_sql, project_id=project_id)

# Convert lists to JSON strings for BigQuery compatibility
product_rows["ingredients"] = product_rows["ingredients"].apply(ensure_list)
product_rows["ingredients"] = product_rows["ingredients"].apply(lambda x: replace_ingredients(x, replacements))
product_rows["ingredients"] = product_rows["ingredients"].apply(lambda x: json.dumps(x) if isinstance(x, list) else x)

# # Apply the replacement to each row's treatments list
print("new df:", product_rows)

# # Upload the updated DataFrame back to BigQuery
table_id = f"{project_id}.skincare_products_stg.sephora_product"
pandas_gbq.to_gbq(product_rows, table_id, project_id=project_id, if_exists='replace')

Downloading: 100%|██████████|
new df:      product_id                                      product_name brand_id  \
0       P474053                                Glossy Nail Polish     1063   
1       P478711                                 Nail Art Stickers     1063   
2       P432502                       BACKSTAGE Glow Face Palette     1073   
3       P457666  J'adore Eau Lumière Eau de Toilette Roller-pearl     1073   
4       P379408                                         Nail Glow     1073   
...         ...                                               ...      ...   
8489    P434223        The Brush Crush Heated Straightening Brush     7004   
8490    P449576      The Tress Press Straightening Iron 1.25 Inch     7004   
8491    P406232                           10X Pro Styling Iron 1"     7086   
8492    P460864                           OnePass 1" Styling Iron     7086   
8493    P442581                        GrapheneMX Styling Iron 1"     7086   

     brand_name  loves_co

100%|██████████| 1/1 [00:00<00:00, 6114.15it/s]


##### Cast ingredients column to array

In [136]:
%%bigquery
SELECT
  -- Use JSON_EXTRACT_ARRAY to convert the cleaned string into an array
  JSON_EXTRACT_ARRAY(ingredients) AS ingredients
FROM
  `skincare_products_stg.sephora_product`

Query is running:   0%|          |

Downloading:   0%|          |

,ingredients
0,"[""alcohol"", ""fragrance"", ""dipropylene glycol"",..."
1,"[""alcohol denat."", ""water"", ""fragrance"", ""ethy..."
2,"[""aqua (water)"", ""butylene glycol"", ""glycerin""..."
3,"[""alcohol"", ""parfum/fragrance"", ""aqua/water"", ..."
4,"[""silica"", ""mica"", ""dimethicone/divinyldimethi..."
...,...
8489,[]
8490,[]
8491,[]
8492,[]


In [137]:
%%bigquery
create or replace table skincare_products_stg.sephora_product as
  select
    product_id,
    product_name,
    brand_id,
    brand_name,
    loves_count,
    rating,
    reviews,
    other_size,
    imperial_size,
    metric_size,
    color,
    variation_count,
    variation_max_price,
    variation_min_price,
    JSON_EXTRACT_ARRAY(ingredients) as ingredients,
    price_usd,
    value_price_usd,
    sale_price_usd,
    limited_edition,
    new_product,
    online_only,
    out_of_stock,
    sephora_exclusive,
    feature_highlights,
    primary_category,
    secondary_category,
    tertiary_category,
    _data_source,
    _load_time
  from skincare_products_stg.sephora_product

Query is running:   0%|          |

""


##### Verify ingredients have been standardized

In [138]:
%%bigquery
WITH flattened_ingredients AS (
  -- Flatten the array into individual rows
  SELECT
    LOWER(TRIM(ingredient)) AS ingredient -- Convert to lowercase and trim spaces to handle case/whitespace variations
  FROM
    skincare_products_stg.sephora_product,
    UNNEST(ingredients) AS ingredient -- Unnest the arrays into individual strings
  WHERE
    ingredients IS NOT NULL
)
-- Select distinct treatments and sort them alphabetically
SELECT DISTINCT ingredient
FROM flattened_ingredients
ORDER BY ingredient;

Query is running:   0%|          |

Downloading:   0%|          |

,ingredient
0,"""#12103/a alcohol"""
1,"""#14881/a talc"""
2,"""#17515:"""
3,"""#17541:"""
4,"""#17627:"""
...,...
18380,"""¬glycyrrhiza glabra (licorice) root extract"""
18381,"""˝adore: calcium aluminum borosilicate"""
18382,"""α-phellandrene"""
18383,"""ψ hydrolyzed quinoa"""


### `sephora_product_review` table

#### Cast `author_id` from STRING to INTEGER

In [86]:
%%bigquery
select author_id as orig_author_id, safe_cast(author_id as INTEGER) as new_author_id
from skincare_products_raw.sephora_product_review
limit 5

Query is running:   0%|          |

Downloading:   0%|          |

,orig_author_id,new_author_id
0,2652440379,2652440379
1,5123288012,5123288012
2,8577823224,8577823224
3,1812815731,1812815731
4,9348546709,9348546709


#### Cast `is_recommended` from FLOAT to BOOL

In [87]:
%%bigquery
select
  is_recommended as orig_is_recommended,
  CASE
    WHEN is_recommended = 1 THEN TRUE
    WHEN is_recommended = 0 THEN FALSE
    ELSE NULL
  END as new_is_recommended
from skincare_products_raw.sephora_product_review
limit 5

Query is running:   0%|          |

Downloading:   0%|          |

,orig_is_recommended,new_is_recommended
0,NaN,<NA>
1,NaN,<NA>
2,NaN,<NA>
3,1.0,True
4,1.0,True


#### Replace values of `Grey` with `Gray` in `eye_color` field

In [88]:
%%bigquery
select
  eye_color as orig_eye_color,
  case eye_color when 'Grey' then 'gray' else eye_color end as new_eye_color
from skincare_products_raw.sephora_product_review
limit 100

Query is running:   0%|          |

Downloading:   0%|          |

,orig_eye_color,new_eye_color
0,None,None
1,None,None
2,None,None
3,brown,brown
4,brown,brown
...,...,...
95,None,None
96,brown,brown
97,None,None
98,None,None


#### Rename `total_feedback_*` fields to something that makes more sense

In [89]:
%%bigquery
select
  total_feedback_count as review_feedback_count,
  total_pos_feedback_count as review_upvote_count,
  total_neg_feedback_count as review_downvote_count,
from skincare_products_raw.sephora_product_review
limit 5

Query is running:   0%|          |

Downloading:   0%|          |

,review_feedback_count,review_upvote_count,review_downvote_count
0,39,12,27
1,7,3,4
2,19,18,1
3,12,1,11
4,68,60,8


#### Drop `row_number` field

In [ ]:
# Leave out row_number during `select` query

#### Putting it all together

In [90]:
%%bigquery
select
  safe_cast(author_id as INTEGER) as author_id,
  rating,
  CASE
    WHEN is_recommended = 1 THEN TRUE
    WHEN is_recommended = 0 THEN FALSE
    ELSE NULL
  END as is_recommended,
  helpfulness,
  total_feedback_count as review_feedback_count,
  total_pos_feedback_count as review_upvote_count,
  total_neg_feedback_count as review_downvote_count,
  submission_time,
  review_text,
  review_title,
  skin_tone,
  case eye_color when 'Grey' then 'gray' else eye_color end as new_eye_color,
  skin_type,
  hair_color,
  product_id,
  product_name,
  brand_name,
  price_usd,
  _data_source,
  _load_time,
from skincare_products_raw.sephora_product_review

Query is running:   0%|          |

Downloading:   0%|          |

,author_id,rating,is_recommended,helpfulness,review_feedback_count,review_upvote_count,review_downvote_count,submission_time,review_text,review_title,skin_tone,new_eye_color,skin_type,hair_color,product_id,product_name,brand_name,price_usd,_data_source,_load_time
0,1533231934,4,<NA>,0.250000,4,1,3,2008-10-25,Murad’s exfoliating cleanser is a good product...,Very good cleanser...,fair,None,oily,None,P4010,AHA/BHA Exfoliating Cleanser,Murad,46.0,sephora_product_review,2024-09-25 02:15:00.273556+00:00
1,9548845160,5,<NA>,0.930233,43,40,3,2017-03-27,I struggled with acne on my forehead and templ...,MY MOST RECOMMENDED PRODUCT,None,None,oily,None,P4016,Acne Control Clarifying Cleanser,Murad,36.0,sephora_product_review,2024-09-25 02:15:00.273556+00:00
2,25285368579,2,False,0.777778,9,7,2,2021-02-12,"I bought the travel size, tried it for like tw...",Dried out my oily skin,lightMedium,brown,oily,brown,P4016,Acne Control Clarifying Cleanser,Murad,36.0,sephora_product_review,2024-09-25 02:15:00.273556+00:00
3,2177603932,1,<NA>,0.200000,5,1,4,2015-03-24,it makes my skins so dry and .. i don’t like t...,do not like the smell,None,None,None,None,P12045,Grape Water Moisturizing Face Mist,Caudalie,12.0,sephora_product_review,2024-09-25 02:15:00.273556+00:00
4,1450844384,1,<NA>,0.272727,11,3,8,2009-10-30,not verey good producet to oily even for dry skin,fresh sugar face polish 4.2 oz,lightMedium,None,dry,None,P12295,Sugar Face Polish Exfoliator,fresh,64.0,sephora_product_review,2024-09-25 02:15:00.273556+00:00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1094406,30165722656,3,False,0.333333,6,2,4,2021-09-06,"After reading about the one day results, I gav...",None,fairLight,blue,dry,brown,P503191,Mini Indigo Overnight Repair Serum in Cream Tr...,Tatcha,22.0,sephora_product_review,2024-09-25 02:15:00.273556+00:00
1094407,40261604063,5,True,0.333333,3,1,2,2022-12-20,I have red spots on my face and just within a ...,A new staple in my routine,fair,hazel,combination,None,P503693,Bouncy Brightfacial Brightening Mask with 10% ...,Drunk Elephant,68.0,sephora_product_review,2024-09-25 02:15:00.273556+00:00
1094408,6908668210,5,True,0.333333,3,1,2,2020-08-12,I received this product complementary for test...,I love it!! Perfect cleanser!,light,brown,normal,black,P504595,Mini Rice Wash Skin-Softening Cleanser,Tatcha,18.0,sephora_product_review,2024-09-25 02:15:00.273556+00:00
1094409,5045348881,3,False,0.333333,6,2,4,2020-08-16,I tried this cleanser a couple of times and I ...,Gross Smell,lightMedium,brown,combination,brown,P504615,Mini Deep Cleanse Gentle Exfoliating Cleanser,Tatcha,18.0,sephora_product_review,2024-09-25 02:15:00.273556+00:00


#### Create Staging table

In [91]:
%%bigquery
create or replace table skincare_products_stg.sephora_product_review as
  select
    safe_cast(author_id as INTEGER) as author_id,
    rating,
    CASE
      WHEN is_recommended = 1 THEN TRUE
      WHEN is_recommended = 0 THEN FALSE
      ELSE NULL
    END as is_recommended,
    helpfulness,
    total_feedback_count as review_feedback_count,
    total_pos_feedback_count as review_upvote_count,
    total_neg_feedback_count as review_downvote_count,
    submission_time,
    review_text,
    review_title,
    skin_tone,
    case eye_color when 'Grey' then 'gray' else eye_color end as new_eye_color,
    skin_type,
    hair_color,
    product_id,
    product_name,
    brand_name,
    price_usd,
    _data_source,
    _load_time,
  from skincare_products_raw.sephora_product_review

Query is running:   0%|          |

""


### `skin_disease_picture` table



##### Replace `'N/A'` with null

In [92]:
%%bigquery
select
case skin_condition_name when 'N/A' then null else skin_condition_name end as skin_condition_name,
case skin_color when 'N/A' then null else skin_color end as skin_color,
case severity when 'N/A' then null else severity end as severity,
case image_link when 'N/A' then null else image_link end as image_link,
  _data_source, _load_time
  from skincare_products_raw.skin_disease_picture;

Query is running:   0%|          |

Downloading:   0%|          |

,skin_condition_name,skin_color,severity,image_link,_data_source,_load_time
0,aids,red,severe,https://storage.cloud.google.com/skincare-prod...,skin_disease_picture,2024-09-25 02:15:36.624486+00:00
1,ctcl,red,severe,https://storage.cloud.google.com/skincare-prod...,skin_disease_picture,2024-09-25 02:15:36.624486+00:00
2,ctcl,red,severe,https://storage.cloud.google.com/skincare-prod...,skin_disease_picture,2024-09-25 02:15:36.624486+00:00
3,ctcl,red,moderate,https://storage.cloud.google.com/skincare-prod...,skin_disease_picture,2024-09-25 02:15:36.624486+00:00
4,ctcl,red,moderate,https://storage.cloud.google.com/skincare-prod...,skin_disease_picture,2024-09-25 02:15:36.624486+00:00
...,...,...,...,...,...,...
3813,vitiligo,"light brown, white",moderate,https://storage.cloud.google.com/skincare-prod...,skin_disease_picture,2024-09-25 02:15:36.624486+00:00
3814,pseudomonas_cellulitis,"red, green, yellow",severe,https://storage.cloud.google.com/skincare-prod...,skin_disease_picture,2024-09-25 02:15:36.624486+00:00
3815,vasculitis,"red, purple, brown",severe,https://storage.cloud.google.com/skincare-prod...,skin_disease_picture,2024-09-25 02:15:36.624486+00:00
3816,pyoderma_gangrenosum,"red, yellow, and purple",severe,https://storage.cloud.google.com/skincare-prod...,skin_disease_picture,2024-09-25 02:15:36.624486+00:00


##### Look for empty strings

In [93]:
%%bigquery
select count(*) as empty_skin_condition_name
from skincare_products_raw.skin_disease_picture
where skin_condition_name = '';

Query is running:   0%|          |

Downloading:   0%|          |

,empty_skin_condition_name
0,0


In [94]:
%%bigquery
select count(*) as empty_skin_color
from skincare_products_raw.skin_disease_picture
where skin_color = '';

Query is running:   0%|          |

Downloading:   0%|          |

,empty_skin_color
0,0


In [95]:
%%bigquery
select count(*) as empty_severity
from skincare_products_raw.skin_disease_picture
where severity = '';

Query is running:   0%|          |

Downloading:   0%|          |

,empty_severity
0,0


In [96]:
%%bigquery
select count(*) as empty_image_link
from skincare_products_raw.skin_disease_picture
where image_link = '';

Query is running:   0%|          |

Downloading:   0%|          |

,empty_image_link
0,0


##### Putting it all together

In [97]:
%%bigquery
select
case skin_condition_name
  when 'N/A' then null
  when '\\N' then null
  when '' then null
  else skin_condition_name
  end as skin_condition_name,
case skin_color
  when 'N/A' then null
  when '\\N' then null
  when '' then null
  else skin_color
  end as affected_skin_color,
case severity
  when 'N/A' then null
  when '\\N' then null
  when '' then null
  else severity
  end as severity,
case image_link
  when 'N/A' then null
  when '\\N' then null
  when '' then null
  else image_link
  end as image_link,
_data_source, _load_time
from skincare_products_raw.skin_disease_picture;

Query is running:   0%|          |

Downloading:   0%|          |

,skin_condition_name,affected_skin_color,severity,image_link,_data_source,_load_time
0,aids,red,severe,https://storage.cloud.google.com/skincare-prod...,skin_disease_picture,2024-09-25 02:15:36.624486+00:00
1,ctcl,red,severe,https://storage.cloud.google.com/skincare-prod...,skin_disease_picture,2024-09-25 02:15:36.624486+00:00
2,ctcl,red,severe,https://storage.cloud.google.com/skincare-prod...,skin_disease_picture,2024-09-25 02:15:36.624486+00:00
3,ctcl,red,moderate,https://storage.cloud.google.com/skincare-prod...,skin_disease_picture,2024-09-25 02:15:36.624486+00:00
4,ctcl,red,moderate,https://storage.cloud.google.com/skincare-prod...,skin_disease_picture,2024-09-25 02:15:36.624486+00:00
...,...,...,...,...,...,...
3813,vitiligo,"light brown, white",moderate,https://storage.cloud.google.com/skincare-prod...,skin_disease_picture,2024-09-25 02:15:36.624486+00:00
3814,pseudomonas_cellulitis,"red, green, yellow",severe,https://storage.cloud.google.com/skincare-prod...,skin_disease_picture,2024-09-25 02:15:36.624486+00:00
3815,vasculitis,"red, purple, brown",severe,https://storage.cloud.google.com/skincare-prod...,skin_disease_picture,2024-09-25 02:15:36.624486+00:00
3816,pyoderma_gangrenosum,"red, yellow, and purple",severe,https://storage.cloud.google.com/skincare-prod...,skin_disease_picture,2024-09-25 02:15:36.624486+00:00


##### Create staging table

In [98]:
%%bigquery
create or replace table skincare_products_stg.skin_disease_picture as
  select
case skin_condition_name
  when 'N/A' then null
  when '\\N' then null
  when '' then null
  else skin_condition_name
  end as skin_condition_name,
case skin_color
  when 'N/A' then null
  when '\\N' then null
  when '' then null
  else skin_color
  end as affected_skin_color,
case severity
  when 'N/A' then null
  when '\\N' then null
  when '' then null
  else severity
  end as severity,
case image_link
  when 'N/A' then null
  when '\\N' then null
  when '' then null
  else image_link
  end as image_link,
_data_source, _load_time
from skincare_products_raw.skin_disease_picture;

Query is running:   0%|          |

""


#### Standardize the skin_color field with LLM

In [99]:
%%bigquery
select skin_condition_name
from skincare_products_raw.skin_disease_picture
group by skin_condition_name
order by skin_condition_name

Query is running:   0%|          |

Downloading:   0%|          |

,skin_condition_name
0,11img017
1,11img018
2,11img019
3,14img010
4,1img011
...,...
515,weathering_nodules
516,white_superficial_onychomycosis
517,x_linked_ichthyosis
518,xanthomas


##### Test skin_color standardization

In [ ]:
import json
import vertexai
from vertexai.generative_models import GenerativeModel
from google.cloud import bigquery

project_id = "segfault-434120"
region = "us-central1"
model_name = "gemini-1.5-flash-001"

prompt = """
Go through this list of affected_skin_color values and look for values that have similar meanings but were given different names.
For example, 'light brown' and 'brownish' could be standardized to 'brown'.
Suggest a standard value, mapping the current one to the new one.
Return the list of original values along with their new values.
Format the results as a json object with the schema: current_value:string, new_value:string.
Do not include any unchanged values in your answer.
Do not include an explanation with your answer.
"""

skin_color_sql = """select distinct affected_skin_color
                    from skincare_products_raw.skin_disease_picture
                    where affected_skin_color is not null
                    order by affected_skin_color"""

skin_disease_pictures_sql = """select *
                               from skincare_products_raw.skin_disease_picture
                               where affected_skin_color is not null"""

bq_client = bigquery.Client()

rows = bq_client.query(skin_color_sql).result()

skin_color_list = [row["affected_skin_color"] for row in rows]
skin_color_str = '\n'.join(skin_color_list)

print("skin_color_str:", skin_color_str)

vertexai.init(project=project_id, location=region)
model = GenerativeModel(model_name)

resp = model.generate_content([skin_color_str, prompt])
resp_text = resp.text.replace("```json", "").replace("```", "").replace("\n", "")

print(resp_text)
skin_color_mappings = json.loads(resp_text)
print("skin_color_mappings:", skin_color_mappings)

replacements = {}
for old, new in skin_color_mappings.items():
    if old != new:
        replacements[old] = new

print("replacements:", replacements)

df = bq_client.query(skin_disease_pictures_sql).to_dataframe()

print("Original df:", df)

df["affected_skin_color"] = df["affected_skin_color"].map(replacements).fillna(df["affected_skin_color"])

print("Updated df:", df)

skin_color_str: black
blue
brown
brown and white
brownish
brownish-purple
dark
dark brown
fair
gray
green
grey
light
light brown
light brown, white
light green
light pink
light skin
light tan
light-brown
light-pink
light-skinned
light_brown
light_pink
light_red
light_skin
light_skin_tone
medium brown
null
orange
pale
pale pink
pale yellow
pale_yellow
pigmented
pink
pink, red
pinkish
pinkish red
pinkish-red
pinkish-white
pinkish_red
purple
red
red and purple
red, brown
red, brown, black
red, brown, white
red, green, yellow
red, purple, brown
red, purple, brown, and black
red, white
red, yellow
red, yellow, and purple
red_and_purple
reddish
reddish-brown
skin tone
tan
unknown
white
white, brown
white, red
yellow
yellowish
yellowish, red
yellowish_green
{"brownish": "brown","brownish-purple": "brown","dark brown": "brown","fair": "light","grey": "gray","light brown": "brown","light brown, white": "brown","light green": "green","light pink": "pink","light skin": "light","light tan": "light

##### Apply affected_skin_color standardization to staging table

In [101]:
import json
import pandas_gbq
import vertexai
from vertexai.generative_models import GenerativeModel, Part
from google.cloud import bigquery

project_id = "segfault-434120"
region = "us-central1"
model_name = "gemini-1.5-flash-001"
prompt = """
Go through this list of affected_skin_color values and look for values that have similar meanings but were given different names.
For example, 'rosacea', 'acne rosacea', and 'rosacea (acne)' could all be standardized to 'rosacea'.
Suggest a standard value, mapping the current one to the new one.
Return the list of original values along with their new values.
Format the results as a json object with the schema: current_value:string, new_value:string.
Do not include any unchanged values in your answer.
Do not include an explanation with your answer.
"""
skin_color_sql = """select distinct affected_skin_color
                    from skincare_products_stg.skin_disease_picture
                    where affected_skin_color is not null
                    order by affected_skin_color"""

skin_disease_picture_sql = """select *
                               from skincare_products_stg.skin_disease_picture
                               where affected_skin_color is not null"""

bq_client = bigquery.Client()
rows = bq_client.query_and_wait(skin_color_sql)

category_list = []
for row in rows:
    category_list.append(row["affected_skin_color"])
category_str = '\n'.join(category_list)

vertexai.init(project=project_id, location=region)
model = GenerativeModel(model_name)
resp = model.generate_content([category_str, prompt])
resp_text = resp.text.replace("```json", "").replace("```", "").replace("\n", "")
print(resp_text)
categories = json.loads(resp_text)
print(type(categories))

replacements = {} # will store the new categories

# categories can be either a dictionary or list type (depending on what LLM decides to do!)
if type(categories) == dict:
    for old, new in categories.items():
        if old == new:
            continue
        else:
            replacements[old] = new

if type(categories) == list:
    for cat_entry in categories:
        if cat_entry['current_category'] == cat_entry['new_category']:
            continue
        else:
            replacements[cat_entry['current_category']] = cat_entry['new_category']

print("replacements:", replacements)

df = bq_client.query_and_wait(skin_disease_picture_sql).to_dataframe()
print("orig df:", df)

df["affected_skin_color"] = df["affected_skin_color"].map(replacements).fillna(df["affected_skin_color"])
print("new df:", df)

table_id = "skincare_products_stg.skin_disease_picture"
pandas_gbq.to_gbq(df, table_id, project_id=project_id, if_exists="replace")

{"brownish": "brown","brownish-purple": "brown","dark brown": "brown","fair": "light","grey": "gray","light brown": "brown","light brown, white": "brown","light green": "green","light pink": "pink","light skin": "light","light tan": "tan","light-brown": "brown","light-pink": "pink","light-skinned": "light","light_brown": "brown","light_pink": "pink","light_red": "red","light_skin": "light","light_skin_tone": "light","medium brown": "brown","pale pink": "pink","pale yellow": "yellow","pale_yellow": "yellow","pink, red": "pink","pinkish": "pink","pinkish red": "pink","pinkish-red": "pink","pinkish-white": "pink","pinkish_red": "pink","red and purple": "red","red, brown": "red","red, brown, black": "red","red, brown, white": "red","red, green, yellow": "red","red, purple, brown": "red","red, purple, brown, and black": "red","red, white": "red","red, yellow": "red","red, yellow, and purple": "red","red_and_purple": "red","reddish": "red","reddish-brown": "red","skin tone": "tan","white, br

100%|██████████| 1/1 [00:00<00:00, 9686.61it/s]


#### Standardize the skin_condition_name field with LLM

##### Test skin_condition_name standardization

In [ ]:
import json
import vertexai
from vertexai.generative_models import GenerativeModel
from google.cloud import bigquery

project_id = "segfault-434120"
region = "us-central1"
model_name = "gemini-1.5-flash-001"

prompt = """
Go through this list of skin_condition_name values and look for values that have similar meanings but were given different names.
For example, 'rosacea', 'acne rosacea', and 'rosacea (acne)' could all be standardized to 'rosacea'.
Suggest a standard value, mapping the current one to the new one.
Return the list of original values along with their new values.
Format the results as a json object with the schema: current_value:string, new_value:string.
Do not include any unchanged values in your answer.
Do not include an explanation with your answer.
"""

skin_condition_name_sql = """select distinct skin_condition_name
                             from skincare_products_stg.skin_disease_picture
                             where skin_condition_name is not null
                             order by skin_condition_name"""

skin_disease_pictures_sql = """select *
                               from skincare_products_stg.skin_disease_picture
                               where skin_condition_name is not null"""

bq_client = bigquery.Client()

rows = bq_client.query(skin_condition_name_sql).result()

skin_condition_name_list = [row["skin_condition_name"] for row in rows]
skin_condition_name_str = '\n'.join(skin_condition_name_list)
print("skin_condition_name_str:", skin_condition_name_str)

vertexai.init(project=project_id, location=region)
model = GenerativeModel(model_name)

resp = model.generate_content([skin_condition_name_str, prompt])
resp_text = resp.text.replace("```json", "").replace("```", "").replace("\n", "")
print(resp_text)

skin_condition_mappings = json.loads(resp_text)
print("skin_condition_mappings:", skin_condition_mappings)

replacements = {}
for old, new in skin_condition_mappings.items():
    if old != new:
        replacements[old] = new
print("replacements:", replacements)

df = bq_client.query(skin_disease_pictures_sql).to_dataframe()
print("Original df:", df)

df["skin_condition_name"] = df["skin_condition_name"].map(replacements).fillna(df["skin_condition_name"])
print("Updated df:", df)

skin_condition_name_str: 11img017
11img018
11img019
14img010
1img011
1img015
3img006
4img005
4th_1img008
4th_disease
7img017
9img008
acanthosis_nigricans
accessory_nipple
accessory_trachus
acne
acne_closed_comedo
acne_cystic
acne_excoriated
acne_infantile
acne_keloidalis
acne_open_comedo
acne_pustular
acne_scar
acquired_digital_fibrokeratoma
acrocyanosis
acrodermatitis_enteropathica
actinic_cheilitis
actinic_cheilitis_sq_cell
actinic_cheilitis_sq_cell_lip
actinic_comedones
actinic_keratosis
actinic_keratosis_5fu
actinic_keratosis_horn
actinic_keratosis_pigmented
actinomycosis
acute_paronychia
aids
albinism
allergic_contact_dermatitis
alopecia_areata
amyloidosis
androgenic_alopecia
angioedema
angiokeratomas
angry_back_patch_testing
arsenical_keratoses
atopic_dermatitis
atopic_lichenification
atrophy_blanche
atypical_mycobacterium
atypical_nevi
axillary_granular_parakeratosis
basal_cell_carcinoma
basal_cell_carcinoma_lid
basal_cell_carcinoma_pigmented
basal_cell_carcinoma_sclerosing
basa

##### Apply skin_condition_name standardization to staging table

In [102]:
import json
import pandas_gbq
import vertexai
from vertexai.generative_models import GenerativeModel, Part
from google.cloud import bigquery

project_id = "segfault-434120"
region = "us-central1"
model_name = "gemini-1.5-flash-001"
prompt = """
Go through this list of skin_condition_name values and look for values that have similar meanings but were given different names.
For example, 'rosacea', 'acne rosacea', and 'rosacea (acne)' could all be standardized to 'rosacea'.
Suggest a standard value, mapping the current one to the new one.
Return the list of original values along with their new values.
Format the results as a json object with the schema: current_value:string, new_value:string.
Do not include any unchanged values in your answer.
Do not include an explanation with your answer.
"""
skin_condition_name_sql = """select distinct skin_condition_name
                             from skincare_products_stg.skin_disease_picture
                             where skin_condition_name is not null
                             order by skin_condition_name"""

skin_disease_picture_sql = """select *
                               from skincare_products_stg.skin_disease_picture
                               where skin_condition_name is not null"""

bq_client = bigquery.Client()
rows = bq_client.query_and_wait(skin_condition_name_sql)

category_list = []
for row in rows:
    category_list.append(row["skin_condition_name"])
category_str = '\n'.join(category_list)

vertexai.init(project=project_id, location=region)
model = GenerativeModel(model_name)
resp = model.generate_content([category_str, prompt])
resp_text = resp.text.replace("```json", "").replace("```", "").replace("\n", "")
print(resp_text)
categories = json.loads(resp_text)
print(type(categories))

replacements = {} # will store the new categories

# categories can be either a dictionary or list type (depending on what LLM decides to do!)
if type(categories) == dict:
    for old, new in categories.items():
        if old == new:
            continue
        else:
            replacements[old] = new

if type(categories) == list:
    for cat_entry in categories:
        if cat_entry['current_category'] == cat_entry['new_category']:
            continue
        else:
            replacements[cat_entry['current_category']] = cat_entry['new_category']

print("replacements:", replacements)

df = bq_client.query_and_wait(skin_disease_picture_sql).to_dataframe()
print("orig df:", df)

df["skin_condition_name"] = df["skin_condition_name"].map(replacements).fillna(df["skin_condition_name"])
print("new df:", df)

table_id = "skincare_products_stg.skin_disease_picture"
pandas_gbq.to_gbq(df, table_id, project_id=project_id, if_exists="replace")

{"acne_closed_comedo": "acne","acne_cystic": "acne","acne_excoriated": "acne","acne_infantile": "acne","acne_keloidalis": "acne","acne_open_comedo": "acne","acne_pustular": "acne","actinic_cheilitis_sq_cell": "actinic cheilitis","actinic_cheilitis_sq_cell_lip": "actinic cheilitis","bowens_disease": "bowen's disease","candidiasis_diaper": "candidiasis","candidiasis_large_skin_folds": "candidiasis","candidiasis_mouth": "candidiasis","candidiasis_of_the_tongue": "candidiasis","chigger_bites_bullous": "chigger bites","dermatitis_areola": "eczema","dermatitis_arm": "eczema","dermatitis_drug": "eczema","dermatitis_herpetiformis": "eczema","dermatitis_lids": "eczema","dermatitis_swimming": "eczema","eczema_acute": "eczema","eczema_areola": "eczema","eczema_areolae": "eczema","eczema_arms": "eczema","eczema_asteatotic": "eczema","eczema_axillae": "eczema","eczema_chronic": "eczema","eczema_ears": "eczema","eczema_fingertips": "eczema","eczema_foot": "eczema","eczema_hand": "eczema","eczema_her

100%|██████████| 1/1 [00:00<00:00, 7681.88it/s]
